# CardIt — Search Engine Demo
Hybrid BM25 + FAISS semantic search over competitive debate evidence cards.

Note: Search Engine HTML outputs show only the tag and initial cite for the card
      This was done to limit the clutter in the notebook
      In real deployment, the full card is viewable with formatting and easily copy/pastable

## 1. Load the Search Engine

In [2]:
import sys
sys.path.insert(0, '..')

from search.search_engine import SearchEngine

# Loads BM25 index, FAISS index, embedding model, and cross-encoder reranker.
# We will be searching through the NDT CEDA 2018 dataset (college policy debate)
engine = SearchEngine(dataset_name='ndtceda18', use_reranker=True)

Loading dataset: ndtceda18
Loading BM25 index...
Loading FAISS index and embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 758.80it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading cross-encoder reranker...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 539.91it/s, Materializing param=classifier.weight]                                    
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Search engine ready.


## 2. Helper — Display Results

In [3]:
from IPython.display import display, HTML

def show_results(response, max_cards=5):
    t = response['timings']
    results = response['results'][:max_cards]

    timing_str = (
        f"BM25 {t.get('bm25', 0)*1000:.0f}ms | "
        f"FAISS {t.get('dense', 0)*1000:.0f}ms | "
        f"Rerank {t.get('rerank', 0)*1000:.0f}ms | "
        f"Total {t.get('total', 0)*1000:.0f}ms"
    )

    html = f"<p style='color:gray;font-size:0.85em'>{timing_str} &mdash; {len(response['results'])} results</p>"

    for i, card in enumerate(results, 1):
        tag      = card.get('tag') or '(no tag)'
        cite     = card.get('cite') or card.get('fullcite') or ''
        summary  = card.get('summary') or card.get('fulltext') or ''
        school   = card.get('schoolDisplayName') or card.get('schoolName') or ''
        tourn    = card.get('tournament') or ''
        side     = card.get('side') or ''
        meta     = ' | '.join(filter(None, [school, tourn, side]))

        summary_preview = (summary[:400] + '...') if len(summary) > 400 else summary

        html += f"""
        <div style='border:1px solid #ddd;border-left:4px solid #0097a7;
                    border-radius:4px;padding:12px 16px;margin-bottom:10px;
                    font-family:sans-serif'>
            <div style='font-weight:bold;font-size:1em;margin-bottom:4px'>#{i} {tag}</div>
            <div style='color:#555;font-size:0.85em;margin-bottom:6px'>{cite}</div>
            <div style='font-size:0.9em;color:#333'>{summary_preview}</div>
            <div style='color:#888;font-size:0.8em;margin-top:8px'>{meta}</div>
        </div>
        """

    display(HTML(html))

## 3. Basic Keyword Query

In [4]:
results = engine.search('nuclear deterrence stability')
show_results(results)

## 4. Semantic Query — No Shared Keywords

The query below uses plain language with no debate jargon. A pure keyword search would miss most of these cards. The hybrid retriever surfaces cards about **miscalculation theory**, **inadvertent escalation**, and **signaling failure** — none of which share exact words with the query.

In [5]:
results = engine.search('countries misunderstand each other and start wars')
show_results(results)

## 5. BM25-Only vs Hybrid — Side by Side

Compare results with and without the dense retriever to show what semantic search adds.

In [6]:
from search.bm25_retriever import BM25Retriever
from config.settings import get_dataset_paths
import sqlite3

paths = get_dataset_paths('opencaselist-2020-2022')
bm25 = BM25Retriever(index_dir=paths['bm25'])

query = 'climate tipping points irreversible'

# BM25 only
bm25_hits = bm25.search(query, top_k=5)
bm25_ids  = [idx + 1 for idx, _ in bm25_hits]

conn = sqlite3.connect(paths['db'])
conn.row_factory = sqlite3.Row
placeholders = ','.join('?' for _ in bm25_ids)
bm25_cards = [dict(r) for r in conn.execute(
    f'SELECT rowid,* FROM cards WHERE rowid IN ({placeholders})', bm25_ids
)]
conn.close()

print('=== BM25 only ===')
show_results({'results': bm25_cards, 'timings': {}}, max_cards=3)

print('=== Hybrid (BM25 + FAISS + rerank) ===')
hybrid = engine.search(query)
show_results(hybrid, max_cards=3)

=== BM25 only ===


=== Hybrid (BM25 + FAISS + rerank) ===


## 6. Latency Breakdown

In [8]:
import pandas as pd

queries = [
    'nuclear deterrence stability',
    'countries misunderstand each other and start wars',
    'climate tipping points irreversible',
    'economic collapse causes conflict',
    'artificial intelligence existential risk',
]

rows = []
for q in queries:
    r = engine.search(q)
    t = r['timings']
    rows.append({
        'query': q[:50],
        'bm25 (ms)':   round(t.get('bm25', 0) * 1000),
        'faiss (ms)':  round(t.get('dense', 0) * 1000),
        'rerank (ms)': round(t.get('rerank', 0) * 1000),
        'total (ms)':  round(t.get('total', 0) * 1000),
        'results':     len(r['results']),
    })

pd.DataFrame(rows).set_index('query')

,bm25 (ms),faiss (ms),rerank (ms),total (ms),results
query,,,,,
nuclear deterrence stability,5,10,213,239,200
countries misunderstand each other and start wars,6,9,223,248,200
climate tipping points irreversible,11,10,441,472,200
economic collapse causes conflict,6,8,77,99,200
artificial intelligence existential risk,11,9,358,602,200
